# 🧹 Data Cleaning Pipeline
## ERP Sales Analytics - Shoebadoo E-Commerce Data

**Objective:**
- Remove duplicates (verification step)
- Handle missing values intelligently
- Extract missing information from product descriptions using NLP
- Calculate derived fields (total_amount)
- Validate and correct data types
- Save cleaned datasets for quality validation

**Based on findings from:** `01_data_exploration.ipynb`

**Key Issues to Address:**
- ⚠️ PRODUCTS: 19.2% NULL in product_name
- ⚠️ PRODUCTS: 29.8% NULL in category
- ⚠️ PRODUCTS: 33.2% NULL in brand

## 1. Setup & Imports

In [ ]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnan, count, sum as spark_sum, 
    regexp_extract, upper, trim, coalesce, lit,
    length, lower, regexp_replace
)
from pyspark.sql.types import *
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich!")

## 2. Spark Session erstellen

In [ ]:
# Spark Session erstellen
spark = SparkSession.builder \
    .appName("ERP-Sales-Data-Cleaning") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"✅ Spark Session erstellt!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

## 3. Daten laden (aus data_exploration)

Wir laden die Daten, die wir im vorherigen Notebook gespeichert haben.

In [ ]:
# Pfad-Konfiguration
INPUT_PATH = "/app/data/cleaned"  # Output vom ersten Notebook
OUTPUT_PATH = "/app/data/cleaned"  # Überschreiben mit finalen sauberen Daten

print("📂 Lade Daten aus:", INPUT_PATH)
print("-" * 80)

try:
    # Lade bereinigte RAW Daten vom ersten Notebook
    customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_raw.parquet")
    print(f"✅ customers_df geladen: {customers_df.count():,} rows")
    
    products_df = spark.read.parquet(f"{INPUT_PATH}/products_raw.parquet")
    print(f"✅ products_df geladen:  {products_df.count():,} rows")
    
    sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_raw.parquet")
    print(f"✅ sales_df geladen:     {sales_df.count():,} rows")
    
    returns_df = spark.read.parquet(f"{INPUT_PATH}/returns_raw.parquet")
    print(f"✅ returns_df geladen:   {returns_df.count():,} rows")
    
    print("\n🎉 Alle Dateien erfolgreich geladen!")
    
except Exception as e:
    print(f"❌ Fehler beim Laden: {e}")
    print("\n💡 Tipp: Hast du das erste Notebook (01_data_exploration.ipynb) ausgeführt?")

## 4. Duplikate entfernen (Verification)

Obwohl wir aus der EDA wissen, dass keine Duplikate vorhanden sind, führen wir dies als Best Practice durch.

In [ ]:
print("🔍 Entferne Duplikate basierend auf Primary Keys...\n")

# Vor der Deduplizierung
customers_before = customers_df.count()
products_before = products_df.count()
sales_before = sales_df.count()
returns_before = returns_df.count()

# Duplikate entfernen basierend auf Primary Keys
customers_clean = customers_df.dropDuplicates(["customer_id"])
products_clean = products_df.dropDuplicates(["product_id"])
sales_clean = sales_df.dropDuplicates(["sale_id"])
returns_clean = returns_df.dropDuplicates(["return_id"])

# Nach der Deduplizierung
customers_after = customers_clean.count()
products_after = products_clean.count()
sales_after = sales_clean.count()
returns_after = returns_clean.count()

# Report
print(f"CUSTOMERS: {customers_before:,} → {customers_after:,} ({customers_before - customers_after} removed)")
print(f"PRODUCTS:  {products_before:,} → {products_after:,} ({products_before - products_after} removed)")
print(f"SALES:     {sales_before:,} → {sales_after:,} ({sales_before - sales_after} removed)")
print(f"RETURNS:   {returns_before:,} → {returns_after:,} ({returns_before - returns_after} removed)")

print("\n✅ Deduplizierung abgeschlossen!")

## 5. Missing Values - Initial Fill

Fülle fehlende Werte mit Platzhaltern (UNKNOWN, 0), die später durch intelligentere Methoden ersetzt werden.

In [ ]:
print("🔧 Fülle fehlende Werte mit Platzhaltern...\n")

# CUSTOMERS (eigentlich keine NULLs, aber als Best Practice)
customers_clean = customers_clean.fillna({
    'first_name': 'UNKNOWN',
    'last_name': 'UNKNOWN',
    'email': 'unknown@example.com',
    'country': 'UNKNOWN'
})
print("✅ CUSTOMERS: Platzhalter gesetzt")

# PRODUCTS (die Tabelle mit den meisten NULL-Werten!)
# Produktname 19.2%, category 29.8%, brand 33.2%
products_clean = products_clean.fillna({
    'product_name': 'UNKNOWN',
    'category': 'UNKNOWN',
    'brand': 'UNKNOWN',
    'price': 0.0
})
print("✅ PRODUCTS: Platzhalter gesetzt (werden später durch NLP ersetzt)")

# SALES
sales_clean = sales_clean.fillna({
    'total_amount': 0.0,
    'channel': 'UNKNOWN',
    'payment_method': 'UNKNOWN'
})
print("✅ SALES: Platzhalter gesetzt")

# RETURNS
returns_clean = returns_clean.fillna({
    'refunded_amount': 0.0,
    'return_reason': 'UNKNOWN'
})
print("✅ RETURNS: Platzhalter gesetzt")

print("\n🎉 Initiale Platzhalter gesetzt!")

## 6. Calculate Missing total_amount (SALES)

Fehlende `total_amount` Werte können berechnet werden: `price * quantity`

In [ ]:
print("💰 Berechne fehlende total_amount Werte...\n")

# Count missing before
missing_before = sales_clean.filter((col("total_amount") == 0) | col("total_amount").isNull()).count()
print(f"Fehlende total_amount Werte: {missing_before:,}")

# Join mit products um Preis zu bekommen
sales_clean = sales_clean.join(
    products_clean.select("product_id", col("price").alias("product_price")),
    on="product_id",
    how="left"
)

# Berechne total_amount: wenn NULL oder 0, dann product_price * quantity
sales_clean = sales_clean.withColumn(
    "total_amount",
    when(
        (col("total_amount").isNull()) | (col("total_amount") == 0),
        col("product_price") * col("quantity")
    ).otherwise(col("total_amount"))
)

# Temporäre Spalte entfernen
sales_clean = sales_clean.drop("product_price")

# Count missing after
missing_after = sales_clean.filter((col("total_amount") == 0) | col("total_amount").isNull()).count()
print(f"Fehlende total_amount nach Berechnung: {missing_after:,}")
print(f"✅ {missing_before - missing_after:,} Werte berechnet!")

print("\n✅ total_amount Berechnung abgeschlossen!")

## 7. NLP: Extract Category, Brand, Product Name from Description

Dies ist der **wichtigste Teil**: Wir extrahieren fehlende Informationen aus der `description` Spalte.

In [ ]:
print("🤖 NLP: Extrahiere Kategorie, Marke und Produktname aus Beschreibung...\n")

# Bekannte Kategorien (aus EDA)
known_categories = ['Sport', 'Schuhe', 'Bekleidung', 'Accessoires', 'NULL']

# Bekannte Marken (aus EDA - erweitere diese Liste nach Bedarf!)
known_brands = [
    'Boss', 'Eastpak', 'Nike', 'Adidas', 'Puma', 'Reebok',
    'New Balance', 'Asics', 'Converse', 'Vans', 'Under Armour',
    'North Face', 'Patagonia', 'Columbia', 'Timberland'
]

# Regex-Pattern für Kategorie-Extraktion
# Sucht nach "Kategorie: XXX" oder ähnlichen Mustern
category_pattern = r'Kategorie[:\s]+([A-Za-zäöüÄÖÜß]+)'

print(f"📋 Bekannte Kategorien: {len(known_categories)}")
print(f"🏷️ Bekannte Marken: {len(known_brands)}")
print(f"🔍 Kategorie Pattern: {category_pattern}")

### 7.1 Extract Category from Description

In [ ]:
print("\n📦 Extrahiere Kategorie aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories vorher: {unknown_before:,}")

# Extract category from description using regex
products_clean = products_clean.withColumn(
    "extracted_category",
    regexp_extract(col("description"), category_pattern, 1)
)

# Update category only where it was UNKNOWN and extraction found something
products_clean = products_clean.withColumn(
    "category",
    when(
        (col("category") == "UNKNOWN") & (length(col("extracted_category")) > 0),
        col("extracted_category")
    ).otherwise(col("category"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_category")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("category") == "UNKNOWN").count()
print(f"UNKNOWN categories nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Kategorien extrahiert!")

### 7.2 Extract Brand from Description

In [ ]:
print("\n🏷️ Extrahiere Marke aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands vorher: {unknown_before:,}")

# Create regex pattern: match any known brand (case-insensitive)
brand_pattern = r'\b(' + '|'.join(known_brands) + r')\b'

# Extract brand from description
products_clean = products_clean.withColumn(
    "extracted_brand",
    regexp_extract(upper(col("description")), brand_pattern.upper(), 1)
)

# Update brand only where it was UNKNOWN and extraction found something
products_clean = products_clean.withColumn(
    "brand",
    when(
        (col("brand") == "UNKNOWN") & (length(col("extracted_brand")) > 0),
        col("extracted_brand")
    ).otherwise(col("brand"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_brand")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("brand") == "UNKNOWN").count()
print(f"UNKNOWN brands nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Marken extrahiert!")

### 7.3 Extract Product Name (first N words from description)

In [ ]:
print("\n📝 Extrahiere Produktname aus description...")

# Count UNKNOWN before
unknown_before = products_clean.filter(col("product_name") == "UNKNOWN").count()
print(f"UNKNOWN product_names vorher: {unknown_before:,}")

# Extract first 3-5 words from description as product name
# Regex: Match first 3-5 words (max 50 chars)
name_pattern = r'^([A-Za-zäöüÄÖÜß0-9\s]{1,50})'

products_clean = products_clean.withColumn(
    "extracted_name",
    trim(regexp_extract(col("description"), name_pattern, 1))
)

# Update product_name only where it was UNKNOWN
products_clean = products_clean.withColumn(
    "product_name",
    when(
        (col("product_name") == "UNKNOWN") & (length(col("extracted_name")) > 0),
        col("extracted_name")
    ).otherwise(col("product_name"))
)

# Drop temporary column
products_clean = products_clean.drop("extracted_name")

# Count UNKNOWN after
unknown_after = products_clean.filter(col("product_name") == "UNKNOWN").count()
print(f"UNKNOWN product_names nachher: {unknown_after:,}")
print(f"✅ {unknown_before - unknown_after:,} Produktnamen extrahiert!")

print("\n🎉 NLP-Extraktion abgeschlossen!")

## 8. Data Type Validation & Correction

Stelle sicher, dass alle Spalten die korrekten Datentypen haben.

In [ ]:
print("🔧 Validiere und korrigiere Datentypen...\n")

# PRODUCTS: Preis muss positiv sein
negative_prices = products_clean.filter(col("price") < 0).count()
print(f"⚠️ Negative Preise gefunden: {negative_prices}")

if negative_prices > 0:
    # Setze negative Preise auf 0 (oder absoluten Wert)
    products_clean = products_clean.withColumn(
        "price",
        when(col("price") < 0, 0).otherwise(col("price"))
    )
    print("✅ Negative Preise auf 0 gesetzt")

# SALES: Menge muss positiv sein
negative_qty = sales_clean.filter(col("quantity") <= 0).count()
print(f"⚠️ Ungültige Mengen gefunden: {negative_qty}")

if negative_qty > 0:
    # Setze ungültige Mengen auf 1
    sales_clean = sales_clean.withColumn(
        "quantity",
        when(col("quantity") <= 0, 1).otherwise(col("quantity"))
    )
    print("✅ Ungültige Mengen auf 1 gesetzt")

# SALES: total_amount muss >= 0 sein
negative_amounts = sales_clean.filter(col("total_amount") < 0).count()
print(f"⚠️ Negative Beträge gefunden: {negative_amounts}")

if negative_amounts > 0:
    sales_clean = sales_clean.withColumn(
        "total_amount",
        when(col("total_amount") < 0, 0).otherwise(col("total_amount"))
    )
    print("✅ Negative Beträge auf 0 gesetzt")

# String Spalten trimmen (remove leading/trailing spaces)
string_columns = [field.name for field in products_clean.schema.fields if isinstance(field.dataType, StringType)]
for col_name in string_columns:
    products_clean = products_clean.withColumn(col_name, trim(col(col_name)))

print("\n✅ Datentyp-Validierung abgeschlossen!")

## 9. Final Quality Check

Prüfe die Datenqualität nach dem Cleaning.

In [ ]:
print("\n" + "="*80)
print(" 📊 FINAL QUALITY CHECK - AFTER CLEANING")
print("="*80 + "\n")

def quick_quality_check(df, name):
    print(f"📦 {name}:")
    total = df.count()
    print(f"   Total Rows: {total:,}")
    
    # Count UNKNOWN values
    for col_name in df.columns:
        unknown_count = df.filter(col(col_name) == "UNKNOWN").count()
        if unknown_count > 0:
            percentage = unknown_count / total * 100
            print(f"   ⚠️ {col_name}: {unknown_count:,} UNKNOWN ({percentage:.1f}%)")
    print()

quick_quality_check(customers_clean, "CUSTOMERS")
quick_quality_check(products_clean, "PRODUCTS")
quick_quality_check(sales_clean, "SALES")
quick_quality_check(returns_clean, "RETURNS")

print("="*80)

## 10. Sample Preview - Verify Cleaning Results

In [ ]:
print("\n👀 PRODUCTS Sample (nach Cleaning):")
products_clean.select("product_id", "product_name", "category", "brand", "price").show(5, truncate=False)

## 11. Save Cleaned Data

Speichere die bereinigten Daten als Parquet für die nächste Phase.

In [ ]:
OUTPUT_PATH = "/app/data/cleaned"

print(f"\n💾 Speichere bereinigte Daten nach: {OUTPUT_PATH}")
print("-" * 80)

try:
    # Überschreibe die alten _raw files mit den bereinigten Daten
    customers_clean.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/customers_clean.parquet")
    print("✅ customers_clean.parquet gespeichert")
    
    products_clean.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/products_clean.parquet")
    print("✅ products_clean.parquet gespeichert")
    
    sales_clean.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/sales_clean.parquet")
    print("✅ sales_clean.parquet gespeichert")
    
    returns_clean.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/returns_clean.parquet")
    print("✅ returns_clean.parquet gespeichert")
    
    print("\n🎉 Alle Dateien erfolgreich gespeichert!")
    
except Exception as e:
    print(f"⚠️ Fehler beim Speichern: {e}")

## 12. Cleaning Summary Report

In [ ]:
print("\n" + "="*80)
print(" 📋 DATA CLEANING SUMMARY")
print("="*80 + "\n")

print("✅ **Completed Tasks:**")
print("   1. Removed duplicates (verification)")
print("   2. Filled missing values with intelligent defaults")
print("   3. Calculated missing total_amount in SALES")
print("   4. Extracted category from product descriptions")
print("   5. Extracted brand from product descriptions")
print("   6. Extracted product names from descriptions")
print("   7. Validated and corrected data types")
print("   8. Saved cleaned datasets")

print("\n📊 **Quality Improvements:**")
print("   - PRODUCTS: Reduced NULL values significantly")
print("   - SALES: Calculated all missing amounts")
print("   - All tables: Validated data types and ranges")

print("\n⏭️ **Next Steps:**")
print("   - Run 03_data_quality_validation.ipynb")
print("   - Implement Great Expectations validation")
print("   - Define data contracts")
print("   - Proceed to dimensional modeling")

print("\n" + "="*80)

In [ ]:
# Spark Session beenden
spark.stop()
print("\n✅ Spark Session beendet. Data Cleaning abgeschlossen!")